# Meeting Notes Agent — Testing Lab

End-to-end testing of the agent pipeline without the web UI.
Each section can be run independently after running **Setup** once.

| Section | What it does |
|---|---|
| 0 Setup | Imports, path, display helpers |
| 1 Config | Load LM Studio + Whisper config, test connection |
| 2 Transcript | Load from text, file, or transcribe audio |
| 3 Pipeline | Run the full agent workflow |
| 4 Step Trace | Inspect each agent step and critique scores |
| 5 Outputs | Summary, action items, domain output, suggestions |
| 6 Baseline | Naive single-call comparison |
| 7 Metrics | ROUGE-1/2/L and BERTScore vs transcript |
| 8 A/B | Compare two workflow configurations |
| 9 Batch | Process all audio files in the library |

---
## 0 · Setup

In [ ]:
import sys, os, json, time
from pathlib import Path
from IPython.display import display, HTML, Markdown

# Add backend to path so all modules resolve
BACKEND = Path("../backend").resolve()
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

def show_md(text: str):
    display(Markdown(text))

def show_json(obj, title: str = ""):
    if title:
        print(f"── {title} " + "─" * max(0, 60 - len(title)))
    print(json.dumps(obj, indent=2, default=str))

def hr(label: str = ""):
    print(("\n" + "─" * 60 + (f"  {label}" if label else "")))

print("Backend path:", BACKEND)
print("Setup complete.")

---
## 1 · Config

In [ ]:
import lm_config, whisper_config

lm_cfg = lm_config.load()
whisper_cfg = whisper_config.load()

print("LM Studio")
print(f"  base_url:           {lm_cfg['base_url']}")
print(f"  model:              {lm_cfg.get('model') or '(auto-select from loaded models)'}")
print(f"  max_tokens:         {lm_cfg['max_tokens']}  ← context window size")
print(f"  max_response_tokens:{lm_cfg['max_response_tokens']}  ← max output per call")
print()
print("Whisper")
print(f"  binary_path: {whisper_cfg.get('binary_path') or '(not set)'}")
print(f"  model:       {whisper_cfg.get('model', 'base')}")
print(f"  model_path:  {whisper_cfg.get('model_path') or '(auto-discover)'}")

In [ ]:
# Test LM Studio connection
import httpx

try:
    r = httpx.get(f"{lm_cfg['base_url'].rstrip('/')}/models", timeout=5)
    models = r.json().get("data", [])
    print(f"✓ LM Studio connected — {len(models)} model(s) loaded")
    for m in models:
        print(f"  • {m['id']}")
except Exception as e:
    print(f"✗ LM Studio not reachable: {e}")
    print("  Start LM Studio and load a model before running pipeline cells.")

---
## 2 · Transcript

Choose **one** of the three options below, then run the cell.

In [ ]:
# ── Option A: paste transcript text directly ───────────────────────────────
TRANSCRIPT_TEXT = """
Paste your transcript here.
""".strip()

# ── Option B: load from a .txt file ───────────────────────────────────────
TRANSCRIPT_FILE = ""  # e.g. "/path/to/transcript.txt"

# ── Option C: select an audio file to transcribe (see next cell) ──────────
AUDIO_DOMAIN = "Education"    # Education | Healthcare | Interview | Project
AUDIO_FILE   = "berkeley_cs_lecture_education.mp3"

# ─────────────────────────────────────────────────────────────────────────
# Set SOURCE = 'A', 'B', or 'C'
SOURCE = "C"
# ─────────────────────────────────────────────────────────────────────────

transcript = ""
segments = []

if SOURCE == "A" and TRANSCRIPT_TEXT:
    transcript = TRANSCRIPT_TEXT
    print(f"Option A — pasted text: {len(transcript):,} chars")
elif SOURCE == "B" and TRANSCRIPT_FILE:
    transcript = Path(TRANSCRIPT_FILE).read_text(encoding="utf-8")
    print(f"Option B — file: {len(transcript):,} chars")
elif SOURCE == "C":
    print(f"Option C — will transcribe {AUDIO_DOMAIN}/{AUDIO_FILE}")
    print("Run the next cell to transcribe.")
else:
    print("No source configured — set SOURCE to 'A', 'B', or 'C'.")

In [ ]:
# Option C — transcribe audio with Whisper
# Skip this cell if SOURCE is 'A' or 'B'

from whisper_utils import run_whisper_file, _find_binary, _find_model

AUDIO_LIBRARY = BACKEND / "tests" / "audio"
audio_path = AUDIO_LIBRARY / AUDIO_DOMAIN / AUDIO_FILE

if not audio_path.exists():
    print(f"✗ File not found: {audio_path}")
    domain_dir = AUDIO_LIBRARY / AUDIO_DOMAIN
    if domain_dir.exists():
        files = [f.name for f in domain_dir.iterdir() if f.is_file() and not f.name.startswith(".")]
        print(f"  Available in {AUDIO_DOMAIN}/: {files or '(empty)'}")
    else:
        print(f"  Domain folder '{AUDIO_DOMAIN}' does not exist.")
else:
    size_mb = audio_path.stat().st_size / 1e6
    print(f"Transcribing: {audio_path.name}  ({size_mb:.1f} MB)")
    print(f"Binary: {_find_binary(whisper_cfg)}")
    print(f"Model:  {_find_model(whisper_cfg)}")
    print("(this may take a few minutes for long recordings...)")

    t0 = time.time()
    result = run_whisper_file(str(audio_path), whisper_cfg)
    elapsed = time.time() - t0

    transcript = result["full_text"]
    segments = result["segments"]
    print(f"\nDone in {elapsed:.1f}s")
    print(f"  {len(transcript):,} chars, {len(segments)} segments, model: {result['model_used']}")
    print(f"\nFirst 400 chars:\n{transcript[:400]}...")

In [ ]:
# Browse audio library
AUDIO_LIBRARY = BACKEND / "tests" / "audio"
print("Audio library contents:")
for domain_dir in sorted(AUDIO_LIBRARY.iterdir()):
    if not domain_dir.is_dir():
        continue
    files = [
        f for f in sorted(domain_dir.iterdir())
        if f.suffix.lower() in {".mp3", ".wav", ".m4a", ".ogg", ".webm"}
    ]
    if files:
        print(f"  {domain_dir.name}/")
        for f in files:
            print(f"    {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"  {domain_dir.name}/  (empty — drop audio files here)")

In [ ]:
# Browse timestamped segments (only available after Option C transcription)
if segments:
    print(f"{len(segments)} segments — showing first 10:")
    for s in segments[:10]:
        start = f"{int(s['start'] // 60):02d}:{s['start'] % 60:05.2f}"
        end   = f"{int(s['end']   // 60):02d}:{s['end']   % 60:05.2f}"
        print(f"  [{start} → {end}]  {s['text']}")
else:
    print("No segments — transcription not yet run, or SOURCE is A/B.")

---
## 3 · Pipeline

Configure domain and optional workflow overrides, then run.

In [ ]:
# ── Pipeline configuration ─────────────────────────────────────────────────
DOMAIN = AUDIO_DOMAIN   # override here: "Education" | "Healthcare" | "Interview" | "Project"

KNOWLEDGE_BASE  = ""  # project knowledge base text
SYSTEM_PROMPT   = ""  # project system prompt
TEMPLATE_PROMPT = ""  # leave empty to use domain default

# Set WORKFLOW_OVERRIDE to a dict to customise, or None for domain defaults
WORKFLOW_OVERRIDE = None
# Example — only run Summarizer with no critique:
# WORKFLOW_OVERRIDE = {
#     "steps": ["Summarizer", "ActionItemExtractor"],
#     "critique_steps": [],
#     "critique_threshold": 7.0,
#     "max_retries": 0,
# }

# ── Show the effective workflow ────────────────────────────────────────────
from agents.orchestrator import DOMAIN_WORKFLOWS, _DEFAULT_WORKFLOW
effective = DOMAIN_WORKFLOWS.get(DOMAIN, _DEFAULT_WORKFLOW)
actual = {**effective, **(WORKFLOW_OVERRIDE or {})}

print(f"Domain:            {DOMAIN}")
print(f"Steps:             {actual['steps']}")
print(f"Critique steps:    {actual.get('critique_steps', [])}")
print(f"Threshold:         {actual.get('critique_threshold', 7)}")
print(f"Max retries:       {actual.get('max_retries', 1)}")
if WORKFLOW_OVERRIDE:
    print("\n⚡ Custom WORKFLOW_OVERRIDE active")

In [ ]:
from agents.lab_runner import run_pipeline

assert transcript, "No transcript loaded — run Section 2 first."

print(f"Running pipeline on {len(transcript):,} char transcript...")
t0 = time.time()

run_result = run_pipeline(
    transcript=transcript,
    domain_name=DOMAIN,
    cfg=lm_cfg,
    knowledge_base=KNOWLEDGE_BASE,
    system_prompt=SYSTEM_PROMPT,
    template_prompt=TEMPLATE_PROMPT,
    workflow_override=WORKFLOW_OVERRIDE,
)

elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s  ({run_result['total_ms']} ms)")
if run_result.get("chunked"):
    print(f"  ↳ Chunked map-reduce — {run_result['chunk_count']} segments processed")
print(f"Extraction steps:  {len([s for s in run_result['steps'] if s['phase'] == 'extraction'])}")
print(f"Critique steps:    {len([s for s in run_result['steps'] if s['phase'] == 'critique'])}")
print(f"Confidence score:  {run_result['confidence_score']}")

---
## 4 · Step Trace

In [ ]:
# High-level trace: one line per step
for step in run_result["steps"]:
    phase  = "🔍" if step["phase"] == "extraction" else "⚖️ "
    status = "✓" if step["status"] == "done" else "✗"
    retry  = f" [attempt {step['attempt']}]" if step.get("attempt", 1) > 1 else ""
    score  = f" score={step['critique_score']:.1f}" if step.get("critique_score") is not None else ""
    tok    = step.get("tokens", {})
    tok_s  = f" ({tok.get('input','?')}→{tok.get('output','?')} tok)" if tok else ""
    print(f"{phase} {status}  {step['name']:<35}{step['duration_ms']:>6} ms{score}{retry}{tok_s}")

In [ ]:
# Drill into a specific step
STEP_INDEX = 0   # change to inspect a different step

step = run_result["steps"][STEP_INDEX]
print(f"Step {STEP_INDEX}: {step['name']}")
print(f"  Phase:    {step['phase']}")
print(f"  Status:   {step['status']}")
print(f"  Duration: {step['duration_ms']} ms")
if step.get("critique_score") is not None:
    print(f"  Score:    {step['critique_score']}")
if step.get("error"):
    print(f"  Error:    {step['error']}")
print()
show_json(step["output"], "Output")

In [ ]:
# All critique issues raised across the run
had_issues = False
for step in run_result["steps"]:
    if step["phase"] == "critique":
        issues = step["output"].get("issues", [])
        score  = step["output"].get("score", "?")
        print(f"Critic → {step['name'].replace('Critic→', '')}  score={score}")
        if issues:
            had_issues = True
            for issue in issues:
                print(f"  • {issue}")
        else:
            print("  (no issues)")
if not had_issues:
    print("No critique issues raised.")

---
## 5 · Outputs

In [ ]:
show_md(f"## Summary\n\n{run_result['summary_text']}")

In [ ]:
items = run_result["action_items"]
if items:
    rows = "\n".join(
        f"| {i+1} | {a['task']} | {a.get('owner', '—')} | {a.get('deadline') or '—'} | {a.get('priority', '—')} |"
        for i, a in enumerate(items)
    )
    show_md(f"## Action Items\n\n| # | Task | Owner | Deadline | Priority |\n|---|------|-------|----------|----------|\n{rows}")
else:
    print("No action items extracted.")

In [ ]:
# Raw domain-agent output
domain_agents = ["InterviewAgent", "LectureAgent", "DecisionLogger"]
found = False
for agent in domain_agents:
    if agent in run_result["results"]:
        found = True
        hr(agent)
        show_json(run_result["results"][agent])
if not found:
    print("No domain-specific agent ran for this domain.")

In [ ]:
show_md(f"## Suggestions\n\n{run_result['suggestions_text']}")

---
## 6 · Naive Baseline

Single LLM call with no structured extraction — gives a baseline to compare against.

In [ ]:
from agents.lab_runner import run_naive

print("Running naive baseline (single call, no agents)...")
t0 = time.time()
naive_summary = run_naive(transcript, DOMAIN, lm_cfg)
print(f"Done in {time.time() - t0:.1f}s")

In [ ]:
show_md(f"## Naive Baseline\n\n{naive_summary}")

In [ ]:
# Side-by-side word count comparison
agent_words = len(run_result["summary_text"].split())
naive_words = len(naive_summary.split())
print(f"Summary length — Agent: {agent_words} words   Naive: {naive_words} words")

agent_actions = len(run_result["action_items"])
print(f"Action items   — Agent: {agent_actions}   Naive: (not extracted)")

---
## 7 · Metrics

ROUGE and BERTScore are computed against the **full untruncated transcript** as the reference
(source-recall approach — no gold summary needed).
Higher = more transcript content captured.

In [ ]:
from agents.lab_runner import compute_rouge

rouge_agent = compute_rouge(run_result["summary_text"], transcript)
rouge_naive = compute_rouge(naive_summary, transcript)

print("ROUGE vs full transcript")
print(f"{'Metric':<10} {'Agent':>8} {'Naive':>8} {'Δ':>8}")
print("─" * 38)
for k in ["rouge1", "rouge2", "rougeL"]:
    a = (rouge_agent or {}).get(k)
    n = (rouge_naive or {}).get(k)
    if a is None or n is None:
        print(f"{k:<10} {'—':>8} {'—':>8} {'—':>8}")
        continue
    delta = round(a - n, 3)
    delta_s = f"+{delta:.3f}" if delta > 0 else f"{delta:.3f}"
    print(f"{k:<10} {a:>8.3f} {n:>8.3f} {delta_s:>8}")

In [ ]:
from agents.lab_runner import compute_bertscore

# First run downloads roberta-large (~500 MB), subsequent runs use cache.
print("Computing BERTScore (may download model on first run)...")
t0 = time.time()
bert_agent = compute_bertscore(run_result["summary_text"], transcript)
bert_naive = compute_bertscore(naive_summary, transcript)
print(f"Done in {time.time() - t0:.1f}s")
print()
print("BERTScore vs full transcript (semantic similarity, roberta-large)")
print(f"{'Metric':<12} {'Agent':>8} {'Naive':>8} {'Δ':>8}")
print("─" * 42)
for k in ["precision", "recall", "f1"]:
    a = (bert_agent or {}).get(k)
    n = (bert_naive or {}).get(k)
    if a is None or n is None:
        print(f"{k:<12} {'—':>8} {'—':>8} {'—':>8}")
        continue
    delta = round(a - n, 3)
    delta_s = f"+{delta:.3f}" if delta > 0 else f"{delta:.3f}"
    print(f"{k:<12} {a:>8.3f} {n:>8.3f} {delta_s:>8}")

In [ ]:
from agents.lab_runner import coverage_score, action_recall, hallucination_check
from tests.fixtures import ALL_FIXTURES

fix = next((f for f in ALL_FIXTURES if f["domain"] == DOMAIN), None)
if fix:
    gt = fix["ground_truth"]
    facts = [f for v in gt.values() if isinstance(v, list) and all(isinstance(x, str) for x in v) for f in v]
    combined = " ".join([
        run_result["summary_text"],
        run_result["suggestions_text"],
        json.dumps(run_result["action_items"]),
    ])
    agent_cov = coverage_score(combined, facts)
    naive_cov = coverage_score(naive_summary, facts)
    ar = action_recall(
        run_result["action_items"],
        gt.get("action_owners", []),
        gt.get("action_tasks", []),
    )
    hallucs = hallucination_check(
        run_result["summary_text"] + " " + run_result["suggestions_text"],
        transcript,
    )
    print(f"Ground truth facts:  {len(facts)}")
    print(f"Coverage — Agent: {agent_cov:.3f}   Naive: {naive_cov:.3f}   Δ: {round(agent_cov - naive_cov, 3):+.3f}")
    print(f"Action recall:       {ar if ar is not None else '—'}")
    print(f"Hallucination words: {len(hallucs)}  {hallucs[:10]}")
else:
    print(f"No built-in fixture for domain '{DOMAIN}' — coverage metrics not available.")
    print("(Coverage uses synthetic ground truth facts from tests/fixtures.py)")

---
## 8 · A/B Comparison

Run the pipeline twice with different configurations and compare results.

In [ ]:
# Store current run as result A
result_A = run_result
naive_A  = naive_summary
label_A  = f"{DOMAIN} — domain default"
print(f"A: {label_A}")
print(f"   Steps: {[s['name'] for s in result_A['steps'] if s['phase'] == 'extraction']}")
print(f"   Confidence: {result_A['confidence_score']}   Time: {result_A['total_ms']} ms")

In [ ]:
# Configure and run pipeline B — edit WORKFLOW_B as needed
WORKFLOW_B = {
    "steps": ["Summarizer", "ActionItemExtractor"],
    "critique_steps": [],
    "critique_threshold": 7.0,
    "max_retries": 0,
}
label_B = "Summarizer + ActionItems, no critique"

print(f"Running B: {label_B}...")
t0 = time.time()
result_B = run_pipeline(
    transcript=transcript,
    domain_name=DOMAIN,
    cfg=lm_cfg,
    knowledge_base=KNOWLEDGE_BASE,
    system_prompt=SYSTEM_PROMPT,
    template_prompt=TEMPLATE_PROMPT,
    workflow_override=WORKFLOW_B,
)
naive_B = run_naive(transcript, DOMAIN, lm_cfg)
print(f"Done in {time.time() - t0:.1f}s   Confidence: {result_B['confidence_score']}")

In [ ]:
# A/B metrics comparison table
rows = []
for label, res, naive in [(label_A, result_A, naive_A), (label_B, result_B, naive_B)]:
    r = compute_rouge(res["summary_text"], transcript) or {}
    b = compute_bertscore(res["summary_text"], transcript) or {}
    rows.append({
        "Config": label,
        "Extraction steps": len([s for s in res["steps"] if s["phase"] == "extraction"]),
        "Critique steps":   len([s for s in res["steps"] if s["phase"] == "critique"]),
        "Confidence":       res["confidence_score"],
        "ROUGE-1":          r.get("rouge1"),
        "ROUGE-2":          r.get("rouge2"),
        "ROUGE-L":          r.get("rougeL"),
        "BERTScore F1":     b.get("f1"),
        "Action items":     len(res["action_items"]),
        "Total ms":         res["total_ms"],
    })

import pandas as pd
df = pd.DataFrame(rows).set_index("Config").T
display(df)

In [ ]:
# Show summaries side by side
hr("A: " + label_A)
print(result_A["summary_text"])
hr("B: " + label_B)
print(result_B["summary_text"])

---
## 9 · Batch Testing

Transcribe and process every audio file in the library. Results are collected in a DataFrame.

In [ ]:
# Configure batch run
BATCH_DOMAIN_FILTER   = None  # restrict to one domain, e.g. "Education", or None for all
BATCH_WORKFLOW_OVERRIDE = None  # or a workflow dict to apply to all files

AUDIO_LIBRARY = BACKEND / "tests" / "audio"
batch_files = []
for domain_dir in sorted(AUDIO_LIBRARY.iterdir()):
    if not domain_dir.is_dir():
        continue
    if BATCH_DOMAIN_FILTER and domain_dir.name != BATCH_DOMAIN_FILTER:
        continue
    for f in sorted(domain_dir.iterdir()):
        if f.suffix.lower() in {".mp3", ".wav", ".m4a", ".ogg", ".webm"}:
            batch_files.append({"domain": domain_dir.name, "path": f, "filename": f.name})

print(f"Files to process: {len(batch_files)}")
for bf in batch_files:
    print(f"  {bf['domain']}/{bf['filename']}  ({bf['path'].stat().st_size / 1e6:.1f} MB)")

In [ ]:
batch_results = []
batch_full    = {}  # domain/filename → full result dict for drill-down

for i, bf in enumerate(batch_files):
    key = f"{bf['domain']}/{bf['filename']}"
    print(f"\n[{i+1}/{len(batch_files)}] {key}")

    # Transcribe
    try:
        t0 = time.time()
        tr = run_whisper_file(str(bf["path"]), whisper_cfg)
        t_w = time.time() - t0
        tx = tr["full_text"]
        print(f"  Transcribed in {t_w:.1f}s  ({len(tx):,} chars, model={tr['model_used']})")
    except Exception as e:
        print(f"  ✗ Transcription failed: {e}")
        batch_results.append({"domain": bf["domain"], "filename": bf["filename"], "error": str(e)})
        continue

    # Run pipeline
    try:
        t0 = time.time()
        res = run_pipeline(
            transcript=tx,
            domain_name=bf["domain"],
            cfg=lm_cfg,
            workflow_override=BATCH_WORKFLOW_OVERRIDE,
        )
        naive = run_naive(tx, bf["domain"], lm_cfg)
        t_p = time.time() - t0
        print(f"  Pipeline in {t_p:.1f}s  ({len(res['steps'])} steps, confidence={res['confidence_score']})")
    except Exception as e:
        print(f"  ✗ Pipeline failed: {e}")
        batch_results.append({"domain": bf["domain"], "filename": bf["filename"], "error": str(e)})
        continue

    # Metrics
    rouge_a = compute_rouge(res["summary_text"], tx) or {}
    rouge_n = compute_rouge(naive, tx) or {}
    bert_a  = compute_bertscore(res["summary_text"], tx) or {}

    row = {
        "domain":          bf["domain"],
        "filename":        bf["filename"],
        "chars":           len(tx),
        "chunked":         res.get("chunked", False),
        "steps":           len([s for s in res["steps"] if s["phase"] == "extraction"]),
        "confidence":      res["confidence_score"],
        "ROUGE-L agent":   rouge_a.get("rougeL"),
        "ROUGE-L naive":   rouge_n.get("rougeL"),
        "ROUGE-L Δ":       round((rouge_a.get("rougeL") or 0) - (rouge_n.get("rougeL") or 0), 3),
        "BERTScore F1":    bert_a.get("f1"),
        "action items":    len(res["action_items"]),
        "total ms":        res["total_ms"],
    }
    batch_results.append(row)
    batch_full[key] = {"result": res, "naive": naive, "transcript": tx}
    print(f"  ROUGE-L: {rouge_a.get('rougeL')} (Δ {row['ROUGE-L Δ']:+})  BERTScore F1: {bert_a.get('f1')}")

print(f"\n{'─'*60}")
print(f"Batch complete — {len(batch_results)} file(s) processed")

In [ ]:
import pandas as pd

display_cols = ["domain", "filename", "chars", "steps", "confidence",
                "ROUGE-L agent", "ROUGE-L naive", "ROUGE-L Δ",
                "BERTScore F1", "action items", "total ms"]
df_batch = pd.DataFrame([{c: r.get(c) for c in display_cols} for r in batch_results])
display(df_batch)

In [ ]:
# Drill into a specific batch result
DRILL_KEY = list(batch_full.keys())[0] if batch_full else None  # change to e.g. "Education/berkeley_cs_lecture_education.mp3"

if DRILL_KEY and DRILL_KEY in batch_full:
    br = batch_full[DRILL_KEY]
    print(f"Inspecting: {DRILL_KEY}")
    show_md(f"## Summary\n\n{br['result']['summary_text']}")
    items = br["result"]["action_items"]
    if items:
        rows = "\n".join(
            f"| {i+1} | {a['task']} | {a.get('owner','—')} | {a.get('deadline') or '—'} | {a.get('priority','—')} |"
            for i, a in enumerate(items)
        )
        show_md(f"## Action Items\n\n| # | Task | Owner | Deadline | Priority |\n|---|------|-------|----------|----------|\n{rows}")
else:
    print("No batch results yet — run the batch cell above first.")

In [ ]:
# Export batch results to CSV
import csv, io

if not df_batch.empty:
    out_path = Path("batch_results.csv")
    df_batch.to_csv(out_path, index=False)
    print(f"Saved to {out_path.resolve()}")
else:
    print("No results to export.")